In [36]:
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel,Field

from dotenv import load_dotenv
load_dotenv()

True

In [37]:
def model_name():
    return ChatOpenAI(model="gpt-4o-mini")

model = model_name()

In [38]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive","neutral","negative"]         
    diagnosis: dict
    response: str

In [39]:
class Sentiment(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="Classify the text as positive, negative, or neutral."
    )

In [40]:
def find_sentiment(state: ReviewState) -> dict:
    review = state["review"]

    structured_model = model.with_structured_output(Sentiment)

    prompt = f"""
Analyze the sentiment of the following review.

Review:
{review}
"""

    result: Sentiment = structured_model.invoke(prompt)

    return {"sentiment": result.sentiment}

In [41]:
def check_sentiment(state: ReviewState) -> Literal["run_diagnosis", "positive_response", "neutral_response"]:
    if state["sentiment"] == "negative":
        return "run_diagnosis"
    elif state["sentiment"] == "positive":
        return "positive_response"
    else:
        return "neutral_response"

In [42]:
def neutral_response(state: ReviewState) -> dict:
    review = state["review"]

    prompt = f"""
Write a short, polite response acknowledging this neutral customer review.
Encourage them to share more feedback so we can improve.

Review:
{review}
"""

    response = model.invoke(prompt)
    return {"response": response.content}

In [43]:
def run_diagnosis(state: ReviewState) -> dict:
    review = state["review"]

    prompt = f"""
The following review is negative. Diagnose the exact problem(s)
the customer is facing based on the review.

Return a short, clear diagnosis.

Review:
{review}
"""

    response = model.invoke(prompt)
    return {"diagnosis": response.content}

In [44]:
def negative_response(state: ReviewState) -> dict:
    diagnosis = state["diagnosis"]

    prompt = f"""
Write a polite, empathetic customer support response addressing this diagnosed issue:

{diagnosis}

Keep it short and professional.
"""

    response = model.invoke(prompt)
    return {"response": response.content}

In [45]:
def positive_response(state: ReviewState) -> dict:
    review = state["review"]

    prompt = f"""
Write a short, warm thank-you response to this positive customer review:

{review}
"""

    response = model.invoke(prompt)
    return {"response": response.content}

In [46]:
builder = StateGraph(ReviewState)

# Nodes
builder.add_node("find_sentiment", find_sentiment)
builder.add_node("run_diagnosis", run_diagnosis)
builder.add_node("negative_response", negative_response)
builder.add_node("positive_response", positive_response)
builder.add_node("neutral_response", neutral_response)

# Edges
builder.add_edge(START, "find_sentiment")

builder.add_conditional_edges(
    "find_sentiment",
    check_sentiment,
    {
        "run_diagnosis": "run_diagnosis",
        "positive_response": "positive_response",
        "neutral_response": "neutral_response"   
    }
)

builder.add_edge("run_diagnosis", "negative_response")
builder.add_edge("negative_response", END)
builder.add_edge("positive_response", END)
builder.add_edge("neutral_response", END)   

graph = builder.compile()

In [47]:
if __name__ == "__main__":
    initial_input = {"review": input("Enter review: ")}

    result = graph.invoke(initial_input)

    print("\n--- WORKFLOW RESULT ---")
    print(f"Sentiment: {result['sentiment']}")

    if result.get("diagnosis"):
        print(f"Diagnosis: {result['diagnosis']}")

    print(f"Response: {result['response']}")


--- WORKFLOW RESULT ---
Sentiment: negative
Diagnosis: Diagnosis: The product is of poor quality, leading to it breaking within a very short period of use (one day). This indicates potential issues with durability or manufacturing defects.
Response: Subject: We’re Here to Help

Dear [Customer’s Name],

Thank you for reaching out to us and sharing your experience. I am truly sorry to hear that the product did not meet your expectations and broke within just one day of use. We understand how disappointing this can be.

We take quality concerns very seriously and would like to assist you with this issue. Please let us know if you would prefer a replacement or a full refund, and we will ensure that it is handled promptly.

Thank you for your patience, and we look forward to resolving this matter for you soon.

Best regards,

[Your Name]  
[Your Position]  
[Company Name]  
[Contact Information]  
